# SAC MARL Notebook: CityLearn Challenge 2023 Phase 2 (3 Buildings)

This notebook is a detailed, notebook-style version of `train.py` for the `citylearn_challenge_2023_phase_2_local_evaluation` environment.

Sections included:
- Visualization of the buildings schema
- Visualization of state space
- Visualization of action space
- Reward function details
- KPI analysis
- SAC multi-agent training loop with TensorBoard logging + CityLearn `ui_exports`


In [ ]:
import json
import os
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ray
import torch
from IPython.display import display

from citylearn.wrappers import (
    ClippedObservationWrapper,
    NormalizedObservationWrapper,
    RLlibMultiAgentEnv,
)
from ray.rllib.algorithms.sac import SACConfig
from ray.rllib.policy.policy import PolicySpec

warnings.filterwarnings('ignore', category=DeprecationWarning)
plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'train.py').exists() and (PROJECT_ROOT.parent / 'train.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

CITYLEARN_VERSION = 'v2.5.0'
DATASET_NAME = 'citylearn_challenge_2023_phase_2_local_evaluation'
LOCAL_SCHEMA_PATH = (
    Path(os.environ.get('LOCALAPPDATA', ''))
    / 'intelligent-environments-lab'
    / 'citylearn'
    / 'Cache'
    / CITYLEARN_VERSION
    / 'datasets'
    / DATASET_NAME
    / 'schema.json'
)
AUTO_DOWNLOAD_DATASET_IF_MISSING = True


def download_dataset_archive(citylearn_version: str, dataset_name: str, schema_path: Path):
    import io
    import zipfile
    import requests

    cache_dir = schema_path.parent
    cache_dir.mkdir(parents=True, exist_ok=True)
    zip_url = f'https://github.com/intelligent-environments-lab/CityLearn/archive/refs/tags/{citylearn_version}.zip'
    response = requests.get(zip_url, stream=True, timeout=120)
    response.raise_for_status()

    dataset_prefix = f"CityLearn-{citylearn_version.lstrip('v')}/data/datasets/{dataset_name}/"
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        matched = [name for name in zf.namelist() if name.startswith(dataset_prefix)]
        if not matched:
            raise FileNotFoundError(f'Dataset folder not found in archive: {dataset_prefix}')

        for member in matched:
            relative = member[len(dataset_prefix):]
            if not relative:
                continue
            dest = cache_dir / relative
            dest.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(dest, 'wb') as dst:
                dst.write(src.read())


if not LOCAL_SCHEMA_PATH.exists() and AUTO_DOWNLOAD_DATASET_IF_MISSING:
    print('Local dataset not found. Downloading CityLearn archive...')
    download_dataset_archive(CITYLEARN_VERSION, DATASET_NAME, LOCAL_SCHEMA_PATH)

if not LOCAL_SCHEMA_PATH.exists():
    raise FileNotFoundError(
        f'Schema not found at {LOCAL_SCHEMA_PATH}. '
        'Set AUTO_DOWNLOAD_DATASET_IF_MISSING=True or cache dataset manually.'
    )

# Change reward here, then rerun this cell.
REWARD_FUNCTION_TYPE = 'citylearn.reward_function.IndependentSACReward'
REWARD_FUNCTION_ATTRIBUTES = {}
FORCE_REWARD_OVERRIDE = True

UI_EXPORT_ROOT = PROJECT_ROOT / 'outputs_mappo' / 'ui_exports'
TENSORBOARD_ROOT = PROJECT_ROOT / 'outputs_mappo' / 'tensorboard'
CHECKPOINT_ROOT = PROJECT_ROOT / 'outputs_mappo' / 'checkpoints'
SCHEMA_OVERRIDE_ROOT = PROJECT_ROOT / 'outputs_mappo' / 'schemas'
for d in [UI_EXPORT_ROOT, TENSORBOARD_ROOT, CHECKPOINT_ROOT, SCHEMA_OVERRIDE_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

with LOCAL_SCHEMA_PATH.open('r', encoding='utf-8') as f:
    base_schema_dict = json.load(f)

# Keep dataset files resolvable when using generated schema copy.
base_schema_dict['root_directory'] = str(LOCAL_SCHEMA_PATH.parent)

active_reward = base_schema_dict.get('reward_function', {})
if FORCE_REWARD_OVERRIDE:
    active_reward = {'type': REWARD_FUNCTION_TYPE, 'attributes': REWARD_FUNCTION_ATTRIBUTES}
    base_schema_dict['reward_function'] = active_reward

GENERATED_SCHEMA_PATH = SCHEMA_OVERRIDE_ROOT / f'{DATASET_NAME}_reward_override.json'
with GENERATED_SCHEMA_PATH.open('w', encoding='utf-8') as f:
    json.dump(base_schema_dict, f, indent=2)

SCHEMA_SOURCE = GENERATED_SCHEMA_PATH

print(f'Schema source: {SCHEMA_SOURCE}')
print(f'Active reward: {active_reward}')
print(f'UI exports directory: {UI_EXPORT_ROOT}')
print(f'TensorBoard root: {TENSORBOARD_ROOT}')
print(f'Checkpoint root: {CHECKPOINT_ROOT}')


In [ ]:
def make_env_config(render_mode='none', render_session_name=None):
    env_kwargs = {
        'schema': SCHEMA_SOURCE,
        'render_mode': render_mode,
        'render_directory': UI_EXPORT_ROOT,
    }
    if render_session_name is not None:
        env_kwargs['render_session_name'] = render_session_name

    return {
        'env_kwargs': env_kwargs,
        'wrappers': [NormalizedObservationWrapper, ClippedObservationWrapper],
    }


def make_env(render_mode='none', render_session_name=None):
    return RLlibMultiAgentEnv(
        make_env_config(render_mode=render_mode, render_session_name=render_session_name)
    )


def unwrap_citylearn_env(env):
    current = env
    while hasattr(current, 'env'):
        current = current.env
    return current


def load_schema_dict(schema_source):
    schema_path = Path(schema_source)
    if schema_path.exists():
        with schema_path.open('r', encoding='utf-8') as f:
            return json.load(f)
    raise FileNotFoundError(
        'Local schema.json not found. Cache the dataset first or set SCHEMA_SOURCE to a local schema file.'
    )


env = make_env()
obs, _ = env.reset()
agent_ids = sorted(obs.keys())
base_env = unwrap_citylearn_env(env)
building_names = [b.name for b in base_env.buildings]

print(f'Agents: {agent_ids}')
print(f'Buildings: {building_names}')


## 1) Visualization of Buildings Schema

Read `schema.json`, extract core per-building attributes, and visualize key capacities/powers.


In [ ]:
schema_dict = load_schema_dict(SCHEMA_SOURCE)

building_rows = []
for building_name, cfg in schema_dict['buildings'].items():
    building_rows.append({
        'building': building_name,
        'cooling_nominal_power_kw': cfg['cooling_device']['attributes']['nominal_power'],
        'dhw_device_nominal_power_kw': cfg['dhw_device']['attributes']['nominal_power'],
        'dhw_storage_capacity_kwh': cfg['dhw_storage']['attributes']['capacity'],
        'battery_capacity_kwh': cfg['electrical_storage']['attributes']['capacity'],
        'battery_nominal_power_kw': cfg['electrical_storage']['attributes']['nominal_power'],
        'pv_nominal_power_kw': cfg['pv']['attributes']['nominal_power'],
    })

building_df = pd.DataFrame(building_rows).sort_values('building').reset_index(drop=True)
display(building_df)


In [ ]:
plot_columns = [
    'cooling_nominal_power_kw',
    'dhw_device_nominal_power_kw',
    'dhw_storage_capacity_kwh',
    'battery_capacity_kwh',
    'battery_nominal_power_kw',
    'pv_nominal_power_kw',
]

ax = building_df.set_index('building')[plot_columns].plot(kind='bar', figsize=(13, 5))
ax.set_title('Per-Building Equipment and Storage Parameters from Schema')
ax.set_ylabel('Value')
ax.set_xlabel('Building')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Reward Function Details

The active reward comes from the generated schema (`SCHEMA_SOURCE`). Update `REWARD_FUNCTION_TYPE` and `REWARD_FUNCTION_ATTRIBUTES` in the config cell to switch reward.


In [ ]:
reward_cfg = schema_dict.get('reward_function', {})
reward_type = reward_cfg.get('type', 'unknown')
reward_attributes = reward_cfg.get('attributes', {})

reward_df = pd.DataFrame({
    'reward_parameter': list(reward_attributes.keys()),
    'value': list(reward_attributes.values()),
})

print(f"Configured reward override type: {REWARD_FUNCTION_TYPE}")
print(f"Configured reward override attributes: {REWARD_FUNCTION_ATTRIBUTES}")
print(f"Reward function type loaded by env: {reward_type}")
if len(reward_df) > 0:
    display(reward_df)
else:
    print('No reward attributes found in schema.')


## 2) Visualization of State Space

Using wrapped environment observations (`NormalizedObservationWrapper` + `ClippedObservationWrapper`).


In [ ]:
def state_feature_group(feature_name):
    if 'pricing' in feature_name:
        return 'pricing'
    if 'irradiance' in feature_name or 'solar' in feature_name:
        return 'solar_weather'
    if 'temperature' in feature_name or 'humidity' in feature_name:
        return 'temperature_humidity'
    if 'storage' in feature_name or 'soc' in feature_name:
        return 'storage_state'
    if 'demand' in feature_name or 'load' in feature_name or 'consumption' in feature_name:
        return 'demand_load'
    if 'hour' in feature_name or 'day' in feature_name or 'month' in feature_name:
        return 'time_calendar'
    if 'occupant' in feature_name or 'power_outage' in feature_name:
        return 'occupancy_outage'
    return 'other'

first_agent = agent_ids[0]
wrapped_observation_names = env.env.observation_names[0]
obs_space = env.observation_space[first_agent]

state_df = pd.DataFrame({
    'feature': wrapped_observation_names,
    'low': obs_space.low,
    'high': obs_space.high,
    'sample_value_t0': obs[first_agent],
})
state_df['range'] = state_df['high'] - state_df['low']
state_df['group'] = state_df['feature'].map(state_feature_group)

print(f'Observation dimension per agent: {obs_space.shape[0]}')
display(state_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

state_df['group'].value_counts().sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='steelblue'
)
axes[0].set_title('State Features by Group')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Group')

axes[1].plot(state_df['feature'], state_df['sample_value_t0'], marker='o', linewidth=1)
axes[1].set_title(f'Initial State Vector Values ({first_agent})')
axes[1].set_ylabel('Value (normalized)')
axes[1].set_xlabel('Feature')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()


## 3) Visualization of Action Space

Inspect action names and bounds per building agent.


In [ ]:
action_names = env.env.action_names[0]
act_space = env.action_space[first_agent]

action_df = pd.DataFrame({
    'action': action_names,
    'low': act_space.low,
    'high': act_space.high,
})
action_df['range'] = action_df['high'] - action_df['low']

print(f'Action dimension per agent: {act_space.shape[0]}')
display(action_df)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hlines(y=action_df['action'], xmin=action_df['low'], xmax=action_df['high'], color='tab:blue', linewidth=3)
ax.scatter(action_df['low'], action_df['action'], color='tab:red', label='low', zorder=3)
ax.scatter(action_df['high'], action_df['action'], color='tab:green', label='high', zorder=3)
ax.set_title('Action Bounds per Control Variable')
ax.set_xlabel('Action value')
ax.set_ylabel('Action')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## 4) KPIs

Run one episode with a baseline random policy and compute CityLearn KPIs.


In [ ]:
def rollout_episode(multi_env, policy_fn=None):
    observations, _ = multi_env.reset()
    episode_rewards = {agent_id: 0.0 for agent_id in observations}

    while not multi_env.terminated:
        if policy_fn is None:
            actions = {agent_id: multi_env.action_space[agent_id].sample() for agent_id in observations}
        else:
            actions = policy_fn(observations)

        observations, rewards, _, _, _ = multi_env.step(actions)
        for agent_id, reward in rewards.items():
            episode_rewards[agent_id] += float(reward)

    return episode_rewards


def compute_kpis(multi_env):
    raw = unwrap_citylearn_env(multi_env).evaluate().copy()
    table = (
        raw.pivot(index='cost_function', columns='name', values='value')
        .sort_index()
        .round(4)
    )
    return raw, table


baseline_env = make_env(render_mode='none')
baseline_rewards = rollout_episode(baseline_env, policy_fn=None)
baseline_raw_kpis, baseline_kpi_table = compute_kpis(baseline_env)

print('Random-policy episode rewards per agent:')
print(baseline_rewards)
display(baseline_kpi_table)

baseline_env.close()

## 5) SAC Training Loop + TensorBoard + CityLearn `ui_exports`

This follows your original SAC setup (shared policy for all agents, old RLlib API stack), adds TensorBoard scalar logging, keeps evaluation exports for CityLearn UI, and saves improved model checkpoints based on evaluation KPI.


In [ ]:
N_ITERATIONS = 100_000
EVAL_INTERVAL = 1_000
NUM_GPUS = 1 if torch.cuda.is_available() else 0

print(f'NUM_GPUS set to: {NUM_GPUS}')

sac_config = (
    SACConfig()
    .api_stack(enable_rl_module_and_learner=False, enable_env_runner_and_connector_v2=False)
    .training(
        replay_buffer_config={
            'type': 'MultiAgentPrioritizedReplayBuffer',
            'capacity': 1_000_000,
            'prioritized_replay_alpha': 0.6,
            'prioritized_replay_beta': 0.4,
            'prioritized_replay_eps': 1e-6,
        }
    )
    .resources(num_gpus=NUM_GPUS)
    .environment(RLlibMultiAgentEnv, env_config=make_env_config(render_mode='none'))
    .multi_agent(
        policies={'shared_policy': PolicySpec()},
        policy_mapping_fn=lambda agent_id, *args, **kwargs: 'shared_policy',
    )
    .framework('torch')
)

algo = sac_config.build()
print('SAC algorithm initialized.')


In [ ]:
try:
    from torch.utils.tensorboard import SummaryWriter
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "tensorboard is required for logging. Install it with: pip install tensorboard"
    ) from exc


def log_scalar(writer, tag, value, step):
    if isinstance(value, (int, float, np.integer, np.floating)) and np.isfinite(value):
        writer.add_scalar(tag, float(value), step)


def get_eval_score(district_kpis_df):
    cost_total_row = district_kpis_df[district_kpis_df['cost_function'] == 'cost_total']
    if len(cost_total_row) > 0 and np.isfinite(cost_total_row.iloc[0]['value']):
        return float(cost_total_row.iloc[0]['value']), 'cost_total'

    valid_values = district_kpis_df['value'].replace([np.inf, -np.inf], np.nan).dropna()
    if len(valid_values) == 0:
        return np.nan, 'undefined'

    return float(valid_values.mean()), 'district_mean'


def extract_checkpoint_path(checkpoint_info):
    if isinstance(checkpoint_info, str):
        return checkpoint_info

    checkpoint_obj = getattr(checkpoint_info, 'checkpoint', None)
    if checkpoint_obj is not None:
        return getattr(checkpoint_obj, 'path', str(checkpoint_obj))

    return str(checkpoint_info)


run_name = f'sac_3_buildings_{datetime.now():%Y%m%d_%H%M%S}'
tb_log_dir = TENSORBOARD_ROOT / run_name
checkpoint_run_dir = CHECKPOINT_ROOT / run_name
checkpoint_run_dir.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(tb_log_dir))

history_rows = []
latest_kpi_table = None
best_eval_score = None
best_eval_metric = None
best_checkpoint_path = None

for i in range(1, N_ITERATIONS + 1):
    result = algo.train()

    env_stats = result.get('env_runners', {})
    mean_reward = env_stats.get('episode_reward_mean')
    num_episodes = env_stats.get('num_episodes')
    trained_steps = result.get('num_steps_trained_this_iter')
    sampled_steps = result.get('num_env_steps_sampled_this_iter')

    row = {
        'iteration': i,
        'episode_reward_mean': mean_reward,
        'num_episodes': num_episodes,
        'num_steps_trained_this_iter': trained_steps,
        'num_env_steps_sampled_this_iter': sampled_steps,
        'eval_score': np.nan,
        'eval_metric': None,
        'is_best_checkpoint': False,
    }

    log_scalar(writer, 'train/episode_reward_mean', mean_reward, i)
    log_scalar(writer, 'train/num_episodes', num_episodes, i)
    log_scalar(writer, 'train/num_steps_trained_this_iter', trained_steps, i)
    log_scalar(writer, 'train/num_env_steps_sampled_this_iter', sampled_steps, i)

    learner_metrics = result.get('learners', {})
    for learner_id, metrics in learner_metrics.items():
        for metric_name in ['policy_loss', 'qf_loss', 'alpha_loss', 'alpha_value', 'td_error']:
            if metric_name in metrics:
                log_scalar(writer, f'learner/{learner_id}/{metric_name}', metrics[metric_name], i)

    if i == 1 or i % 10 == 0:
        print(
            f"[Train] Iter {i:05d} | episodes={num_episodes} | mean_reward={mean_reward} | steps_trained={trained_steps}"
        )

    if i % EVAL_INTERVAL == 0 or i == 1:
        session_name = f'eval_ep{i:05d}'
        eval_env = make_env(render_mode='end', render_session_name=session_name)

        eval_obs, _ = eval_env.reset()
        while not eval_env.terminated:
            eval_actions = {
                agent_id: algo.compute_single_action(
                    agent_obs,
                    policy_id='shared_policy',
                    explore=False,
                )
                for agent_id, agent_obs in eval_obs.items()
            }
            eval_obs, _, _, _, _ = eval_env.step(eval_actions)

        raw_kpis, latest_kpi_table = compute_kpis(eval_env)
        district_kpis = raw_kpis[raw_kpis['level'] == 'district']
        eval_score, eval_metric = get_eval_score(district_kpis)

        row['eval_score'] = eval_score
        row['eval_metric'] = eval_metric

        log_scalar(writer, 'eval/score', eval_score, i)
        for _, kpi_row in district_kpis.iterrows():
            log_scalar(writer, f"kpi/{kpi_row['cost_function']}", kpi_row['value'], i)

        improved = np.isfinite(eval_score) and (
            best_eval_score is None or eval_score < best_eval_score
        )

        if improved:
            checkpoint_info = algo.save(checkpoint_dir=str(checkpoint_run_dir))
            checkpoint_path = extract_checkpoint_path(checkpoint_info)

            best_eval_score = float(eval_score)
            best_eval_metric = eval_metric
            best_checkpoint_path = checkpoint_path
            row['is_best_checkpoint'] = True

            print(
                f"[Checkpoint Saved] iter={i:05d} | {eval_metric}={eval_score:.6f} | path={checkpoint_path}"
            )

        eval_env.close()
        print(f"[Exported CityLearn UI data] {UI_EXPORT_ROOT / session_name}")

    history_rows.append(row)

writer.flush()
writer.close()

train_history = pd.DataFrame(history_rows)
display(train_history.tail(10))

if latest_kpi_table is not None:
    print('Latest evaluation KPIs:')
    display(latest_kpi_table)

print(f'TensorBoard logs saved at: {tb_log_dir}')
print(f'Checkpoint directory: {checkpoint_run_dir}')
if best_checkpoint_path is not None:
    print(f'Best checkpoint ({best_eval_metric}={best_eval_score:.6f}): {best_checkpoint_path}')
else:
    print('No checkpoint saved yet (no valid evaluation score found).')
print(f'Run TensorBoard: tensorboard --logdir "{TENSORBOARD_ROOT}"')


In [ ]:
# Cleanup (safe to run multiple times).
try:
    algo.stop()
except Exception:
    pass

try:
    env.close()
except Exception:
    pass

try:
    ray.shutdown()
except Exception:
    pass

print('Cleanup done.')
